# Pre-fit and post-fit distributions

Reconstructed $(\log_{10} Q^2, p_n)$ spectra before and after each fit, in the style of the event distributions of `5_publication_plots.ipynb`. Everything is read from PROfit's `<fit>_v1_PROfile.root`, which the `profile` stage writes:

* **Pre-fit**: the central-value prediction (`*_cv1d`, `*_cv1d_y`, `*_cv_slice_ybinN`) with the pre-fit error band (`*_preerrband*`, systematic throws around the CV, MC statistics included).
* **Post-fit**: PROfit's *constrained* best fit (`*_bestfit*`: spline best fit plus the data-constrained pull of the covariance-type systematics) with the post-fit band (`*_posterrband*`, MCMC throws of the fitted parameters around the best fit plus the data-constrained posterior of the covariance-type systematics).
* **Data**: the fitted sample (`*_data*`), NuWro fake data or beam-on open data depending on the family.

For each fit two figures are written to `figs/<family>_fit_results/<fit>/`: the fully projected $\log_{10} Q^2$ and $p_n$ spectra (`prefit_postfit_projections.pdf`) and the $\log_{10} Q^2$ spectra in slices of $p_n$ (`prefit_postfit_slices.pdf`). Every panel carries a ratio to the pre-fit prediction underneath.

**$\chi^2$ annotation.** Only the global post-fit value is drawn. It is PROfit's minimum $\chi^2$ from `profile.log`, *including* pulls, split exactly into three parts:

$$\chi^2_\mathrm{min} = \underbrace{r^T C^{-1} r}_{\mathrm{stat}} + \underbrace{\hat\alpha^T\hat\alpha}_{\mathrm{syst}} + \underbrace{\hat\theta^T\hat\theta}_{\mathrm{pull}},$$

with $r = d - \tilde\mu$ the residual of the drawn constrained best fit (`bestfit2d`) against the Neyman data statistics $C=\mathrm{diag}(d)$ (zero-data bins excluded), $\hat\alpha$ the posterior pulls of the covariance-type systematic modes (`covmode_pulls`, MC statistics included) and $\hat\theta$ the fitted parameters (flat-prior ones carry no pull). The split is an exact identity at the best fit (Woodbury), checked against the log for every fit. The total is the goodness of fit of the drawn curve; the stat term alone is not calibrated to $N$ (its expectation is $\mathrm{tr}[C(\Sigma+C)^{-1}]$). Do **not** weight the drawn residual with the post-fit band covariance: that is biased low.

The pre-fit $\chi^2$ (data against the CV with the total PROplot covariance, needs a `plot --with-covar` run) is computed differently and is reported only in the summary table, not on the figures.


In [ ]:
from pathlib import Path
import re

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from IPython.display import display

mpl.rcParams.update({
    "figure.dpi": 140,
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.linewidth": 1.1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "legend.frameon": False,
    "mathtext.fontset": "stix",
})

In [ ]:
# Inputs. The constants mirror 5_publication_plots.ipynb and 3_fit_chi2_summary.ipynb so the
# header, the POT and the chi^2 bookkeeping agree with the other figures.
FIT_RESULTS_ROOT = Path("/nevis/hopper/data/epelaez/axial_mass")
CHANNEL = "nu_uBooNE_numuCC1p"
MC_POT = 9.3221e19  # beam-on POT the MC is normalized to (open-data sum, pot_dic.csv)

SUITES = {
    # NuWro fake data, normalized to 1.75e21 POT in the XML; real beam-on data at MC_POT.
    "nuwro": dict(suite="nuwro", data_label="NuWro", target_pot=1.75e21, header="Simulation"),
    "opendata": dict(suite="opendata", data_label="Open data", target_pot=MC_POT, header="Preliminary"),
}

# Fits to draw, in display order, and their legend names (FIT_ORDER of 3_fit_chi2_summary.ipynb).
FITS = [
    "ma", "ma_uniform", "minerva_k8", "minerva_k7",
    "minerva_k6", "lqcd_k7", "lqcd_k6", "minerva_lqcd_k7", "minerva_lqcd_k6",
    "minerva_k6_nuisance", "lqcd_k6_nuisance", "minerva_lqcd_k6_nuisance",
    "minerva_k6_uniform", "minerva_k6_uniform_nuisance",
]
FIT_LABELS = {
    "ma": r"$M_A$ Gaussian prior",
    "ma_uniform": r"$M_A$ uniform prior",
    "minerva_k8": r"MINERvA prior, $k_{\max}=8$",
    "minerva_k7": r"MINERvA prior, $k_{\max}=7$",
    "minerva_k6": r"MINERvA prior, $k_{\max}=6$",
    "lqcd_k6": r"LQCD prior, $k_{\max}=6$",
    "lqcd_k7": r"LQCD prior, $k_{\max}=7$",
    "minerva_lqcd_k7": r"MINERvA+LQCD prior, $k_{\max}=7$",
    "minerva_lqcd_k6": r"MINERvA+LQCD prior, $k_{\max}=6$",
    "minerva_k6_nuisance": r"MINERvA prior w/ nuisance, $k_{\max}=6$",
    "lqcd_k6_nuisance": r"LQCD prior w/ nuisance, $k_{\max}=6$",
    "minerva_lqcd_k6_nuisance": r"MINERvA+LQCD prior w/ nuisance, $k_{\max}=6$",
    "minerva_k6_uniform": r"Uniform prior, $k_{\max}=6$",
    "minerva_k6_uniform_nuisance": r"Uniform prior w/ nuisance, $k_{\max}=6$",
}
# Parameters with a flat prior contribute no pull term to PROfit's best-fit chi^2.
UNIFORM_PARAMETERS = {
    "ma_uniform": {"MACCQE"},
    "minerva_k6_uniform": {"FAzexpMinervaK6PCA1", "FAzexpMinervaK6PCA2"},
    "minerva_k6_uniform_nuisance": {"FAzexpMinervaK6PCA1", "FAzexpMinervaK6PCA2"},
}
CHI2_PATTERN = re.compile(r"Global Best Fit chi(?:\^?2|²):\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)")

search_root = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in (search_root, *search_root.parents)
     if (path / "python" / "notebooks").is_dir() and (path / "xml").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate the ma_zexp repository root")
FIGURE_ROOT = REPO_ROOT / "figs"


def fit_directory(cfg, fit):
    return FIT_RESULTS_ROOT / f"{cfg['suite']}_fit_results" / fit

In [ ]:
# Style shared with 5_publication_plots.ipynb. The pre-fit prediction keeps that notebook's
# hatched grey "total uncertainty" band; the post-fit prediction is the one coloured element.
BACKGROUND = "#FFFFFF"
FOREGROUND = "#303642"
ACCENT = "#020202"
PREFIT_COLOR = FOREGROUND
PREFIT_FACECOLOR = mpl.colors.to_rgba(FOREGROUND, 0.30)
PREFIT_HATCHCOLOR = mpl.colors.to_rgba(FOREGROUND, 0.55)
PREFIT_HATCH = "/////"
POSTFIT_COLOR = "#2a78d6"
POSTFIT_FACECOLOR = mpl.colors.to_rgba(POSTFIT_COLOR, 0.35)
FIGURE_RC = {
    "font.family": "sans-serif", "mathtext.fontset": "dejavusans", "hatch.linewidth": 0.5,
}
Q2_LABEL = r"$\log_{10}\!\left(Q^2_{\mathrm{reco}}/\mathrm{GeV}^2\right)$"
PN_LABEL = r"$p_n^{\mathrm{reco}}$ [GeV]"
# Ratio panels are clipped to this window so one outlier bin does not flatten the rest.
RATIO_LIMITS = (0.4, 1.6)


def _edge_decimals(edges, minimum=1, maximum=4):
    for decimals in range(minimum, maximum + 1):
        if np.allclose(np.round(edges, decimals), edges, rtol=0, atol=1e-9):
            return decimals
    return maximum


def _style_axis(ax, labelsize=None):
    ax.set_facecolor(BACKGROUND)
    ax.tick_params(which="both", direction="in", top=True, right=True,
                   colors=FOREGROUND, labelcolor=FOREGROUND, length=6)
    if labelsize is not None:
        ax.tick_params(labelsize=labelsize)
    for spine in ax.spines.values():
        spine.set_color(FOREGROUND)
        spine.set_linewidth(1.0)


def _band(ax, edges, lower, upper, *, facecolor, edgecolor, hatch, zorder):
    lower = np.clip(lower, 0, None) if hatch else lower
    ax.fill_between(
        edges, np.r_[lower, lower[-1]], np.r_[upper, upper[-1]], step="post",
        facecolor=facecolor, edgecolor=edgecolor, hatch=hatch, linewidth=0, zorder=zorder,
    )

In [ ]:
def _graph_band(graph):
    """(centre, lower, upper) of a PROfit TGraphAsymmErrors band."""
    centre = graph.values(axis="y")
    return centre, centre - graph.errors("low", axis="y"), centre + graph.errors("high", axis="y")


def _read_spectrum(profile_file, suffix):
    """CV, best fit, data and both bands of one PROfit histogram family (``suffix`` is '' for
    the first-variable projection, '_y' for the second, '_slice_ybinN' for a slice)."""
    cv = profile_file[f"{CHANNEL}_cv1d{suffix}" if suffix in ("", "_y") else f"{CHANNEL}_cv{suffix}"]
    bestfit = profile_file[f"{CHANNEL}_bestfit{suffix}"]
    data = profile_file[f"{CHANNEL}_data{suffix}"]
    _, pre_lower, pre_upper = _graph_band(profile_file[f"{CHANNEL}_preerrband{suffix}"])
    post_centre, post_lower, post_upper = _graph_band(profile_file[f"{CHANNEL}_posterrband{suffix}"])
    # Current PROfit anchors the post-fit band on the drawn best fit (data-constrained); older
    # outputs did the same without the constraint. Either way the band graph is the reference.
    if not np.allclose(post_centre, bestfit.values(), rtol=1e-3, atol=1e-3):
        print(f"  note: post-fit band centre differs from the best-fit histogram for {suffix or 'x'}; "
              "drawing the band centre as the post-fit prediction")
    return dict(
        edges=np.round(cv.axis().edges(), 6), prefit=cv.values(), postfit=post_centre,
        data=data.values(), data_error=data.errors(),
        prefit_lower=pre_lower, prefit_upper=pre_upper,
        postfit_lower=post_lower, postfit_upper=post_upper,
    )


def _prefit_chi2(plot_path, data_2d):
    """Neyman chi^2 of the data against the CV with the total pre-fit covariance, as in notebook 5.
    None when the PROplot file has no Covariance directory (run without --with-covar)."""
    if not plot_path.is_file():
        return None
    with uproot.open(plot_path) as plot_file:
        if "Covariance/collapsed_total_frac_cov" not in plot_file:
            return None
        fractional = plot_file["Covariance/collapsed_total_frac_cov"].values()
        cv = plot_file[f"ErrorBand/Var0/{CHANNEL}_cv2d"].values().reshape(-1)
    data = data_2d.reshape(-1)  # PROfit flattens with the first variable as the slow index
    covariance = fractional * np.outer(cv, cv) + np.diag(data)
    residual = data - cv
    return float(residual @ np.linalg.solve(covariance, residual))


def _postfit_split(profile_file, data_2d, chi2_post):
    """Split PROfit's chi2_min into (stat, syst, param). stat: Neyman residual of the drawn
    constrained best fit (bestfit2d) against data statistics, zero-data bins excluded as in the
    metric; syst: squared posterior pulls of the covariance-type systematic modes
    (covmode_pulls); param: Gaussian pulls of the fitted parameters. stat + syst equals PROfit's
    total minus param exactly (Woodbury); the mismatch is printed as a check."""
    if chi2_post is None or "covmode_pulls" not in profile_file:
        return chi2_post
    data = data_2d.reshape(-1)
    bestfit = profile_file[f"{CHANNEL}_bestfit2d"].values().reshape(-1)
    use = data > 0
    stat = float(np.sum((data[use] - bestfit[use]) ** 2 / data[use]))
    syst = float(np.sum(profile_file["covmode_pulls"].values(axis="y") ** 2))
    mismatch = stat + syst - chi2_post["data"]
    if abs(mismatch) > 0.5:
        print(f"  warning: stat + syst differs from PROfit's data term by {mismatch:.2f}")
    return dict(chi2_post, stat=stat, syst=syst)


def _profit_chi2(log_path, fit, parameters):
    """PROfit's global minimum chi^2 split as (total, data term, pull term). The pull term is the
    Gaussian penalty of the fitted parameters (flat-prior ones carry none), as in
    3_fit_chi2_summary.ipynb. The total is the goodness of fit of the drawn constrained best fit."""
    if not log_path.is_file():
        return None
    matches = CHI2_PATTERN.findall(log_path.read_text(errors="replace"))
    if not matches:
        return None
    uniform = UNIFORM_PARAMETERS.get(fit, set())
    pull = sum(value ** 2 for name, value in parameters.items() if name not in uniform)
    total = float(matches[-1])
    return dict(total=total, data=total - pull, pull=pull)


def load_fit_spectra(cfg, fit):
    """Everything the two figures need for one (family, fit); None when the fit has not run."""
    directory = fit_directory(cfg, fit)
    profile_path = directory / f"{fit}_v1_PROfile.root"
    if not profile_path.is_file():
        print(f"{cfg['suite']}/{fit}: {profile_path.name} missing, skipped")
        return None
    with uproot.open(profile_path) as profile_file:
        cv_2d = profile_file[f"{CHANNEL}_cv2d"]
        # PROfit writes single-precision axes; round so 0.1 does not print as 0.1000000015.
        q2_edges, pn_edges = (np.round(axis.edges(), 6) for axis in cv_2d.axes)
        data_2d = profile_file[f"{CHANNEL}_data2d"].values()
        projections = {"q2": _read_spectrum(profile_file, ""), "pn": _read_spectrum(profile_file, "_y")}
        n_pn_bins = len(pn_edges) - 1
        if f"{CHANNEL}_bestfit_slice_ybin1" in profile_file:
            slices = [_read_spectrum(profile_file, f"_slice_ybin{pn_bin + 1}") for pn_bin in range(n_pn_bins)]
        else:
            print(f"{cfg['suite']}/{fit}: no slice histograms in {profile_path.name}; slices figure skipped")
            slices = None
        global_fit = profile_file["global_fit_result"]
        parameters = dict(zip(global_fit.axis().labels(), global_fit.values()))
        one_sigma = profile_file["one_sigma_errs"]
        errors_low = one_sigma.errors("low", axis="y")
        errors_high = one_sigma.errors("high", axis="y")

    n_bins = data_2d.size
    chi2_pre = _prefit_chi2(directory / f"{fit}_v1_PROplot.root", data_2d)
    chi2_post = _profit_chi2(directory / "profile.log", fit, parameters)
    with uproot.open(profile_path) as profile_file:
        chi2_post = _postfit_split(profile_file, data_2d, chi2_post)
    return dict(
        cfg=cfg, fit=fit, label=FIT_LABELS.get(fit, fit), q2_edges=q2_edges, pn_edges=pn_edges,
        projections=projections, slices=slices, n_bins=n_bins,
        chi2_pre=chi2_pre, chi2_post=chi2_post, parameters=parameters,
        parameter_errors=dict(zip(parameters, zip(errors_low, errors_high))),
    )

In [ ]:
def _draw_spectrum(ax, spec):
    """Pre-fit (hatched band, dashed CV), post-fit (filled band, solid line) and data."""
    edges = spec["edges"]
    centres = 0.5 * (edges[:-1] + edges[1:])
    _band(ax, edges, spec["prefit_lower"], spec["prefit_upper"], facecolor=PREFIT_FACECOLOR,
          edgecolor=PREFIT_HATCHCOLOR, hatch=PREFIT_HATCH, zorder=4)
    ax.stairs(spec["prefit"], edges, baseline=None, color=PREFIT_COLOR, linestyle="--",
              linewidth=1.2, zorder=5)
    _band(ax, edges, spec["postfit_lower"], spec["postfit_upper"], facecolor=POSTFIT_FACECOLOR,
          edgecolor="none", hatch=None, zorder=6)
    ax.stairs(spec["postfit"], edges, baseline=None, color=POSTFIT_COLOR, linestyle="-",
              linewidth=1.6, zorder=7)
    ax.errorbar(
        centres, spec["data"], xerr=0.5 * np.diff(edges), yerr=spec["data_error"],
        fmt="o", color="black", markerfacecolor="black", markersize=3.8, elinewidth=1.0,
        capsize=2.0, linestyle="none", zorder=10,
    )
    ax.set_xlim(edges[[0, -1]])
    upper = max(spec["prefit_upper"].max(), spec["postfit_upper"].max(),
                (spec["data"] + spec["data_error"]).max())
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=5, integer=True))
    return upper


def _draw_ratio(ax, spec):
    """The same three ingredients divided by the pre-fit prediction."""
    edges = spec["edges"]
    centres = 0.5 * (edges[:-1] + edges[1:])
    reference = spec["prefit"]
    safe = np.where(reference > 0, reference, np.nan)
    ratio = lambda values: values / safe
    _band(ax, edges, ratio(spec["prefit_lower"]), ratio(spec["prefit_upper"]),
          facecolor=PREFIT_FACECOLOR, edgecolor=PREFIT_HATCHCOLOR, hatch=PREFIT_HATCH, zorder=4)
    ax.axhline(1.0, color=PREFIT_COLOR, linestyle="--", linewidth=1.0, zorder=5)
    _band(ax, edges, ratio(spec["postfit_lower"]), ratio(spec["postfit_upper"]),
          facecolor=POSTFIT_FACECOLOR, edgecolor="none", hatch=None, zorder=6)
    ax.stairs(ratio(spec["postfit"]), edges, baseline=None, color=POSTFIT_COLOR, linewidth=1.6, zorder=7)
    ax.errorbar(
        centres, ratio(spec["data"]), xerr=0.5 * np.diff(edges), yerr=ratio(spec["data_error"]),
        fmt="o", color="black", markerfacecolor="black", markersize=3.8, elinewidth=1.0,
        capsize=2.0, linestyle="none", zorder=10,
    )
    extremes = np.concatenate([
        ratio(spec["data"] - spec["data_error"]), ratio(spec["data"] + spec["data_error"]),
        ratio(spec["prefit_lower"]), ratio(spec["prefit_upper"]),
        ratio(spec["postfit_lower"]), ratio(spec["postfit_upper"]),
    ])
    extremes = extremes[np.isfinite(extremes)]
    half_width = 1.15 * max(abs(extremes - 1.0).max() if extremes.size else 0.2, 0.1)
    half_width = min(half_width, 1.0 - RATIO_LIMITS[0], RATIO_LIMITS[1] - 1.0)
    ax.set_ylim(1.0 - half_width, 1.0 + half_width)
    ax.set_xlim(edges[[0, -1]])
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=4))
    ax.grid(axis="y", color="#9AA4B2", alpha=0.22, linewidth=0.7)


def _legend_entries(data_label):
    handles = [
        (mpl.patches.Patch(facecolor=PREFIT_FACECOLOR, edgecolor=PREFIT_COLOR, hatch=PREFIT_HATCH,
                           linestyle="--", linewidth=1.2)),
        (mpl.patches.Patch(facecolor=POSTFIT_FACECOLOR, edgecolor=POSTFIT_COLOR, linewidth=1.6)),
        mpl.lines.Line2D([], [], color="black", marker="o", linestyle="none", markersize=4.0),
    ]
    labels = ["Pre-fit prediction, total unc.", "Post-fit prediction, post-fit unc.", data_label]
    return handles, labels


def _chi2_text(st):
    parts = []
    n = st["n_bins"]
    # Only the post-fit chi^2 is drawn; the pre-fit value is built differently and lives in the table.
    post = st["chi2_post"]
    if post is not None and "stat" in post:
        parts.append(rf"$\chi^2_{{\mathrm{{post}}}}/N = ({post['stat']:.1f}_{{\,\mathrm{{stat}}}} + "
                     rf"{post['syst']:.1f}_{{\,\mathrm{{syst}}}} + {post['pull']:.1f}_{{\,\mathrm{{pull}}}})"
                     rf"/{n} = {post['total']:.1f}/{n}$")
    elif post is not None:
        parts.append(rf"$\chi^2_{{\mathrm{{post}}}}/N = {post['total']:.1f}/{n}$")
    return "\n".join(parts)


def _header(fig, st, header_y, left_x):
    cfg = st["cfg"]
    fig.text(left_x, header_y, "MicroBooNE", color=ACCENT, fontsize=14, weight="bold", va="bottom")
    fig.text(left_x + 0.105, header_y, cfg["header"], color=ACCENT, fontsize=14, va="bottom")
    fig.text(0.5, header_y, st["label"], color=ACCENT, fontsize=14, ha="center", va="bottom")
    fig.text(0.975, header_y, f"{cfg['target_pot']:.2e} POT".replace("+", ""), color=ACCENT,
             fontsize=14, ha="right", va="bottom")


def _save(fig, st, name):
    directory = FIGURE_ROOT / f"{st['cfg']['suite']}_fit_results" / st["fit"]
    directory.mkdir(parents=True, exist_ok=True)
    fig.savefig(directory / name, format="pdf", dpi=600, bbox_inches="tight", facecolor="white")

In [ ]:
def plot_projections(st):
    """log10 Q^2 and p_n projections with ratio panels; writes prefit_postfit_projections.pdf."""
    panels = [(st["projections"]["q2"], Q2_LABEL), (st["projections"]["pn"], PN_LABEL)]
    with mpl.rc_context(FIGURE_RC):
        fig = plt.figure(figsize=(13.0, 6.4), facecolor=BACKGROUND)
        grid = fig.add_gridspec(2, 2, height_ratios=[3.0, 1.15], hspace=0.06, wspace=0.16,
                                left=0.07, right=0.975, bottom=0.11, top=0.90)
        for column, (spec, xlabel) in enumerate(panels):
            ax = fig.add_subplot(grid[0, column])
            ax_ratio = fig.add_subplot(grid[1, column], sharex=ax)
            upper = _draw_spectrum(ax, spec)
            ax.set_ylim(0, 1.45 * upper)
            _draw_ratio(ax_ratio, spec)
            ax.tick_params(labelbottom=False)
            _style_axis(ax)
            _style_axis(ax_ratio)
            ax_ratio.set_xlabel(xlabel, color=FOREGROUND, fontsize=16, labelpad=5)
            ax.set_ylabel("Events / bin", color=FOREGROUND, fontsize=16)
            ax_ratio.set_ylabel("Ratio to pre-fit", color=FOREGROUND, fontsize=12)
            if column == 0:
                handles, labels = _legend_entries(st["cfg"]["data_label"])
                legend = ax.legend(handles, labels, loc="upper left", bbox_to_anchor=(0.025, 0.975),
                                   frameon=False, fontsize=10, labelspacing=0.4, handlelength=1.6,
                                   borderaxespad=0)
                for text in legend.get_texts():
                    text.set_color(FOREGROUND)
            chi2_text = _chi2_text(st) if column == 1 else ""
            if chi2_text:
                ax.text(0.975, 0.965, chi2_text, transform=ax.transAxes, ha="right", va="top",
                        fontsize=11, color=FOREGROUND, linespacing=1.4)
        _header(fig, st, header_y=0.915, left_x=0.07)
        _save(fig, st, "prefit_postfit_projections.pdf")
        plt.show()


def plot_slices(st):
    """log10 Q^2 spectra in p_n slices with ratio panels; writes prefit_postfit_slices.pdf."""
    if st["slices"] is None:
        return
    slices, pn_edges = st["slices"], st["pn_edges"]
    n_pn_bins = len(slices)
    n_cols = 4
    n_rows = -(-n_pn_bins // n_cols)
    pn_decimals = _edge_decimals(pn_edges)

    with mpl.rc_context(FIGURE_RC):
        fig = plt.figure(figsize=(13.0, 5.0 * n_rows), facecolor=BACKGROUND)
        outer = fig.add_gridspec(n_rows, n_cols, hspace=0.16, wspace=0.10,
                                 left=0.06, right=0.975, bottom=0.07, top=0.865)
        main_axes, ratio_axes = [], []
        for pn_bin, spec in enumerate(slices):
            row, column = divmod(pn_bin, n_cols)
            inner = outer[row, column].subgridspec(2, 1, height_ratios=[3.0, 1.15], hspace=0.05)
            ax = fig.add_subplot(inner[0])
            ax_ratio = fig.add_subplot(inner[1], sharex=ax)
            upper = _draw_spectrum(ax, spec)
            ax.set_ylim(0, 1.35 * upper)
            _draw_ratio(ax_ratio, spec)
            ax.tick_params(labelbottom=False)
            _style_axis(ax, labelsize=10)
            _style_axis(ax_ratio, labelsize=10)
            ax.text(
                0.95, 0.95,
                rf"${pn_edges[pn_bin]:.{pn_decimals}f} \leq p_n^\mathrm{{reco}} "
                rf"< {pn_edges[pn_bin + 1]:.{pn_decimals}f}$ GeV",
                transform=ax.transAxes, ha="right", va="top", fontsize=10, color=FOREGROUND,
            )
            if column == 0:
                ax.set_ylabel("Events / bin", color=FOREGROUND, fontsize=13)
                ax_ratio.set_ylabel("Ratio", color=FOREGROUND, fontsize=11)
            if row == n_rows - 1 or pn_bin + n_cols >= n_pn_bins:
                ax_ratio.set_xlabel(Q2_LABEL, color=FOREGROUND, fontsize=13, labelpad=4)
            main_axes.append(ax)
            ratio_axes.append(ax_ratio)

        handles, labels = _legend_entries(st["cfg"]["data_label"])
        chi2_text = _chi2_text(st).replace("\n", ",  ")
        if chi2_text:
            handles.append(mpl.patches.Patch(facecolor="none", edgecolor="none"))
            labels.append(chi2_text)
        legend = fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.91),
                            ncol=len(labels), frameon=False, fontsize=11, handlelength=1.6,
                            columnspacing=1.8, borderaxespad=0)
        for text in legend.get_texts():
            text.set_color(FOREGROUND)
        _header(fig, st, header_y=0.935, left_x=0.06)
        _save(fig, st, "prefit_postfit_slices.pdf")
        plt.show()

## All families and fits

Edit `FITS` above (or `FAMILIES` here) to change what is drawn. `FITS` currently lists every fit of the production suites. Fits whose `PROfile.root` does not exist yet are reported and skipped. The best-fit parameters below are PROfit's standardized fit coordinates with the profile-scan $1\sigma$ intervals.

In [ ]:
FAMILIES = ["nuwro", "opendata"]

records = []
for family in FAMILIES:
    cfg = SUITES[family]
    for fit in FITS:
        st = load_fit_spectra(cfg, fit)
        if st is None:
            continue
        post = st["chi2_post"] or {}
        nan = float("nan")
        print(f"{family}/{fit}: {st['n_bins']} bins, chi2 pre = {st['chi2_pre']:.1f}, chi2 post = "
              f"{post.get('stat', nan):.1f} stat + {post.get('syst', nan):.1f} syst + "
              f"{post.get('pull', nan):.2f} pull = {post.get('total', nan):.1f}")
        plot_projections(st)
        plot_slices(st)
        for name, value in st["parameters"].items():
            low, high = st["parameter_errors"][name]
            records.append(dict(family=family, fit=fit, parameter=name, best_fit=value,
                                minus=low, plus=high, chi2_pre=st["chi2_pre"],
                                chi2_post=(st["chi2_post"] or {}).get("total"),
                                chi2_post_stat=(st["chi2_post"] or {}).get("stat"),
                                chi2_post_syst=(st["chi2_post"] or {}).get("syst"),
                                chi2_post_pull=(st["chi2_post"] or {}).get("pull"),
                                n_bins=st["n_bins"]))

summary = pd.DataFrame(records)
display(summary.style.format({"best_fit": "{:.3f}", "minus": "{:.3f}", "plus": "{:.3f}",
                              "chi2_pre": "{:.1f}", "chi2_post": "{:.1f}", "chi2_post_stat": "{:.1f}", "chi2_post_syst": "{:.1f}", "chi2_post_pull": "{:.2f}"}, na_rep="n/a"))